# Guided computational reproduction of Willsey et al. (2025)

**Student workbook**  
Paper: *A high-performance brain-computer interface for finger decoding and quadcopter game control in an individual with paralysis*  
DOI: `10.1038/s41591-024-03341-8`

This is a computational reproduction course, not an independent repetition of the implanted experiment. You will work from the released code and data, reimplement key functions, reproduce quantitative panels, and explain every major analytical choice. The capstone goal is to turn that genuine technical ownership into an honest, evidence-backed application portfolio for the Willsey Lab or a closely related neural-engineering group.

Recommended pace: **10 technical weeks at 6-8 hours per week, followed by a one-week application capstone**. Do not start a module until its "before coding" answers are complete, and do not package application claims until the technical defense is complete.

## Learning contract and AI-use policy

AI tools are allowed, but every retained AI suggestion must be independently checked and explained orally. For every module:

1. Write your prediction and function contract **before** requesting or writing code.
2. At the end of each module, paste any exported AI conversation and/or list the exact search terms used. If neither was used, state that directly.
3. Independently test any code or factual claim retained from external assistance.
4. Be prepared to reconstruct or modify one randomly selected function without AI.
5. Keep one git commit per module. Code appearing only in a final bulk commit is not sufficient evidence of process.

Exported AI conversations are private course evidence by default. Do not place them in a public portfolio unless every participant has consented and all personal or sensitive content has been removed. A public AI-use note may summarize what assistance was used and how it was independently checked.

A correct plot with an incorrect explanation is incomplete. A polished AI-generated answer that you cannot defend orally does not satisfy the checkpoint.

## Ten technical weeks plus application capstone

Plan for approximately **6-8 hours each week**. Week 8 also includes an optional overnight full-channel computation after the reduced implementation passes. Do not advance past a checkpoint until the listed evidence is committed.

| Week | Modules and focus | Evidence due by the end of the week | Checkpoint |
|---:|---|---|---|
| 1 | Setup, Modules 0-1: isolated environment, scope, preregistration, provenance | Environment commands and audit; claim map; preregistration table; code/data hashes | Demonstrate the selected kernel and explain computational versus experimental replication |
| 2 | Module 2: MAT schema, clocks, interpolation, velocity | Data dictionary; alignment function; known-answer interpolation tests | Pass an unseen timestamp edge case and explain all shapes and units |
| 3 | Module 3: target changes, trial segmentation, trajectories | Trial-boundary tests; traceable reproduction of Fig. 1c | Trace one plotted interval back to its source rows and repair an endpoint mutation |
| 4 | Module 4A: block and aggregate behavioral metrics | Tested metric functions; Fig. 1e values and plot; discrepancy notes | Predict the effect of changing the hold interval before running it |
| 5 | Module 4B: statistics and unit-of-analysis sensitivity | Reproduced tests plus one block- or day-level sensitivity analysis | Explain nesting, SEM convention, effect size, and p-value limitations |
| 6 | Module 5: participation-ratio dimensionality | Synthetic rank tests; session values; Fig. 2a | Compute two participation ratios by hand and diagnose condition imbalance |
| 7 | Module 6: released decoder and cross-decoder comparison | Tensor-shape audit; deterministic inference; Fig. 2c-d | Diagnose an evaluation-mode mutation and explain correlation limitations |
| 8 | Module 7: directional SNR and channel-count scaling | Reduced seeded sweep; leakage checks; Fig. 3b-c; optional full sweep | Demonstrate train-only fitting and predict the shuffled/subset controls |
| 9 | Modules 8-9: open-loop classification and flight path | Confusion matrices, shuffled-label control, Fig. 4c, coordinate audit | Explain chance performance and repair a coordinate-sign or indexing mutation |
| 10 | Module 10: clean rerun, discrepancy report, and synthesis | All figures and result files regenerated; module AI/search records; final verdict | Twenty-minute technical oral defense and live modification of two selected functions |
| 11 | Module 11: Willsey Lab application-readiness capstone | Public-safe portfolio, technical brief, five-minute talk, CV entry, inquiry draft, evidence-to-claim map, two-minute pitch, and follow-up proposal | Thirty-minute mock research interview and application-language audit |

**Weekly routine:** answer the before-coding questions, commit predictions and pseudocode, implement and test, compare only after tests pass, record AI assistance and discrepancies, then complete the checkpoint.

## Student record

- Name: **[write here]**
- Start date: **[write here]**
- Operating system / computer: **[write here]**
- Environment method (`conda` or `venv`): **[write here]**
- Python environment name/path: **[write here]**
- Python executable reported by the notebook: **[write here]**
- Repository commit used: **[write here]**
- Data archive SHA-256: **[write here]**
- Career / research goal: **[write here]**
- Initial Willsey Lab fit hypothesis: **[write here before the technical work]**
- Official lab source and access date: **[write here]**

**Initial prediction:** Which result do you expect to be hardest to reproduce, and why?  
**Answer:** [write 3-5 sentences before opening the authors' notebook]

## Setup

Creating an isolated Python environment is part of this assignment. Before running the next cell:

1. Choose Conda or Python `venv`; do not use a system interpreter or the Conda base environment.
2. Follow the corresponding cross-platform instructions in `README.md` using `environment.yml` or `requirements.txt`.
3. Record the exact creation, activation, installation, kernel-registration, and notebook-launch commands below.
4. Run `python -m pip check`, select the newly registered Jupyter kernel, and preserve the output or any installation error.
5. If a pinned version is unavailable on your platform, obtain instructor approval before substituting a version and record the change.

Put this notebook anywhere inside the replication project. It searches upward for `work/replication/Data`; if your checkout has another root, edit only `PROJECT_ROOT_OVERRIDE`.

**Chosen environment method:** [write here]  
**Exact commands and `pip check` result:** [write here]

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import interpolate, stats
from scipy.io import loadmat
import sklearn

PROJECT_ROOT_OVERRIDE = None  # Example: Path("path/to/willsey-replication")

def locate_project_root(start=None):
    """Find the nearest parent containing the released data directory."""
    if PROJECT_ROOT_OVERRIDE is not None:
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not (candidate / "work/replication/Data").is_dir():
            raise FileNotFoundError(f"Data directory not found under {candidate}")
        return candidate
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "work/replication/Data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find work/replication/Data. Set PROJECT_ROOT_OVERRIDE."
    )

environment_marker = os.environ.get("CONDA_DEFAULT_ENV") or os.environ.get("VIRTUAL_ENV")
print("operating system:", platform.platform())
print("python version:", sys.version.split()[0])
print("python executable:", sys.executable)
print("environment marker:", environment_marker or "not detected")
print("numpy/scipy/sklearn:", np.__version__, scipy.__version__, sklearn.__version__)
if environment_marker in (None, "base"):
    print("CHECK: confirm that this kernel belongs to your isolated course environment.")

PROJECT_ROOT = locate_project_root()
DATA_ROOT = PROJECT_ROOT / "work/replication/Data"
RELEASED_CODE = PROJECT_ROOT / "work/code_audit"
STUDENT_OUTPUT = PROJECT_ROOT / "outputs/student_replication"
STUDENT_OUTPUT.mkdir(parents=True, exist_ok=True)

print("project:", PROJECT_ROOT)
print("data:", DATA_ROOT)

**Setup questions**

1. Why did you choose Conda or `venv`, and what tradeoff does the other option have?
2. How do `sys.executable` and the selected Jupyter kernel demonstrate that the isolated environment is actually being used?
3. Why is installing packages into a system interpreter or Conda base environment poor reproducibility practice?
4. What problem does `locate_project_root` solve that a fixed absolute path does not?
5. Why should generated figures go outside the downloaded data directory?
6. Which package-version differences are most likely to change numerical results or random partitions?
7. What evidence would distinguish "the notebook ran" from "the analysis reproduced"?

**Answers:** [write here]

# Module 0 - Read, scope, and preregister

**Learning objective:** Separate the scientific experiment, released offline analysis, and claims that cannot be tested from the public artifacts.

Read the abstract, Figs. 1-4, Methods headings, Data availability, and Code availability. Do not inspect the authors' analysis functions yet.

**Before coding**

1. State the paper's four central quantitative claims in your own words.
2. Identify the observational unit for each claim: participant, session, block, trial, time bin, channel subset, or flight.
3. Which conclusions would require access to the live acquisition and decoder-retraining system?
4. Why is rerunning released code not an independent replication of the biological experiment?
5. Name two risks of treating trials nested within a day as independent observations.

**Answers:** [write 1-2 paragraphs]

### Preregistration table

Fill this before computing results.

| Target | Input data | Primary statistic | Expected tolerance | Failure criterion |
|---|---|---|---|---|
| Fig. 1e performance | [write] | [write] | [write] | [write] |
| Fig. 2a dimensionality | [write] | [write] | [write] | [write] |
| Fig. 2d cross-decoder | [write] | [write] | [write] | [write] |
| Fig. 3c channel scaling | [write] | [write] | [write] | [write] |
| ED Fig. 1b-c classifier | [write] | [write] | [write] | [write] |
| Fig. 4c flight exemplar | [write] | [write] | [write] | [write] |

### Module 0 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 0 (5 minutes):** Explain the boundary between computational reproduction and experimental replication without using the words "same" or "different." Show your preregistration commit.

# Module 1 - Provenance and data integrity

**Learning objective:** Verify exactly which code and bytes you analyzed.

Implement a streaming SHA-256 function. It must work on files larger than memory and return lowercase hexadecimal text.

**Function contract questions**

- Purpose of the function: [write]
- Inputs and output type: [write]
- Invariant maintained across chunks: [write]
- Time and memory complexity: [write]
- One failure mode that should raise an exception: [write]

In [ ]:
def sha256_file(path, chunk_bytes=1024 * 1024):
    """Return the SHA-256 digest of a file without loading it all at once."""
    # TODO: validate path and chunk_bytes, update a hashlib.sha256 object,
    # and return the hexadecimal digest.
    raise NotImplementedError

In [ ]:
# Synthetic self-check: create a tiny file whose known digest is stable.
tiny = STUDENT_OUTPUT / "sha_test.txt"
tiny.write_bytes(b"abc")
assert sha256_file(tiny) == (
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)
print("SHA-256 self-check passed")

In [ ]:
data_files = sorted(path for path in DATA_ROOT.rglob("*") if path.is_file())
inventory = pd.DataFrame({
    "relative_path": [str(p.relative_to(DATA_ROOT)) for p in data_files],
    "bytes": [p.stat().st_size for p in data_files],
    "suffix": [p.suffix.lower() for p in data_files],
})
display(inventory.groupby("suffix").agg(files=("relative_path", "count"), bytes=("bytes", "sum")))
display(inventory.head())

**Interpretation questions**

1. How many MAT files and PyTorch checkpoints are present?
2. Why is a file count insufficient without hashes?
3. Record the Git commit and Dryad version. Which one identifies code and which identifies data?
4. If two students obtain the same rounded plot values but different archive hashes, what should happen next?
5. Explain why `statsMatlab.mat` may be an analysis output rather than missing source data.

**Answers:** [write here]

### Module 1 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 1:** Give the student a second file with one altered byte. Ask them to identify it from a manifest without opening either file.

# Module 2 - MAT schema, clocks, and interpolation

**Learning objective:** Understand how task streams and neural streams are aligned before any metric is computed.

Start with the block used for the 2-DOF trajectory example.

In [ ]:
EXAMPLE_2D = DATA_ROOT / (
    "20230321/RedisMat/"
    "t5_t5.2023.03.21_Data_RedisMat_20230321_162754_(18).mat"
)
raw = loadmat(EXAMPLE_2D)

def mat_manifest(mapping):
    """Return a table of non-metadata MAT variables, shapes, dtypes and bytes."""
    # TODO: ignore keys beginning with '__' and return a DataFrame.
    raise NotImplementedError

display(mat_manifest(raw))

**Schema questions**

1. Which arrays hold estimated position, target position, neural features, and their clocks?
2. Which dimension represents time in each array?
3. Why are the task and neural arrays not assumed to have the same timestamps?
4. What are the units of the Redis clocks and the 50-ms bins?
5. What evidence suggests that 192 physical electrodes are represented inside 256-wide arrays?

**Answers:** [write here]

In [ ]:
def align_stream_to_clock(source_time, source_values, destination_time):
    """Linearly interpolate one time-by-feature stream onto another clock."""
    # TODO: validate monotonic timestamps, check time is axis 0, reject
    # extrapolation, and return an array with len(destination_time) rows.
    raise NotImplementedError

In [ ]:
# Synthetic interpolation test. The underlying relation is y = [2t, -t].
source_t = np.array([0.0, 1.0, 2.0])
source_y = np.column_stack([2 * source_t, -source_t])
destination_t = np.array([0.25, 0.5, 1.5])
expected = np.array([[0.5, -0.25], [1.0, -0.5], [3.0, -1.5]])
np.testing.assert_allclose(
    align_stream_to_clock(source_t, source_y, destination_t), expected
)
print("Interpolation self-check passed")

**Implementation questions**

1. Why must interpolation occur before differentiating position to obtain velocity?
2. What is the consequence of allowing extrapolation beyond the task clock?
3. How would duplicate timestamps break a linear interpolator?
4. The released `LoadData` assigns `binTrial` using the target interpolator. Why does that not affect the manuscript analyses here, and why is it still a bug?
5. Draw the shapes of all inputs and outputs to `align_stream_to_clock`.

**Answers:** [write here]

### Module 2 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 2:** Introduce an unsorted timestamp and an out-of-range destination time. The student must predict and then demonstrate the behavior of their function.

# Module 3 - Targets, trial boundaries, and trajectory plots

**Learning objective:** Reconstruct Fig. 1c from one released block and explain every transformation.

Implement detection of target transitions. The first row is always a trial-start candidate; subsequent starts occur when any target coordinate changes.

In [ ]:
def transition_indices(targets, atol=0.0):
    """Return row indices where a multivariate target first appears or changes."""
    # TODO: validate a 2-D input, compare consecutive rows across columns,
    # and include index 0 exactly once.
    raise NotImplementedError

In [ ]:
synthetic_targets = np.array([
    [0.5, 0.5],
    [0.5, 0.5],
    [1.0, 0.5],
    [1.0, 0.5],
    [0.5, 0.5],
    [0.5, 0.0],
])
np.testing.assert_array_equal(
    transition_indices(synthetic_targets), np.array([0, 2, 4, 5])
)
print("Transition self-check passed")

In [ ]:
FIG1C_BLOCK = DATA_ROOT / (
    "20230413/RedisMat/"
    "t5_t5.2023.04.13_Data_RedisMat_20230413_153321_(45).mat"
)

def load_aligned_position_and_target(path, finger_columns=(0, 1, 2, 5)):
    """Load a block and align position and selected targets to neural time."""
    # TODO: use the fields identified in Module 2. Restrict neural times
    # to the task-clock range before interpolation.
    raise NotImplementedError

def plot_target_trajectories(time_ms, position, targets, target_half_width):
    """Create four stacked trajectory axes for the first 100 seconds."""
    # TODO: draw decoded position and target rectangles/steps. Label units.
    raise NotImplementedError

# aligned = load_aligned_position_and_target(FIG1C_BLOCK)
# fig = plot_target_trajectories(**aligned)
# fig.savefig(STUDENT_OUTPUT / "fig1c_replication.png", dpi=180, bbox_inches="tight")

**Figure questions**

1. Why does the paper show positions from -1 to 1 while the notebook export uses 0-100%?
2. Which transformations change appearance but not information?
3. How is target width obtained from `fn_hold_threshold`?
4. Why is elapsed time measured from the first retained neural sample?
5. The authors' notebook later saves block 48 under the block-45 filename. How would you detect this overwrite from execution order and hashes?
6. Show one intermediate plot that would reveal a clock-alignment error before the final figure is made.

**Answers:** [write here]

### Module 3 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 3:** Point to any 10-second interval in the student's plot. Ask them to identify the target, decoded trace, transition indices, and the corresponding source-array rows.

# Module 4 - Behavioral metrics and Fig. 1e

**Learning objective:** Reimplement trial-level metrics without importing the authors' notebook.

Required functions:

- `sem_population(values)`: notebook convention, `ddof=0`.
- `path_efficiency(path)`: straight-line displacement divided by traveled distance.
- `analyze_block(path, finger_columns)`: successful and analyzed trials plus metrics.
- `aggregate_blocks(results)`: means, SEMs, completion, and per-block rate.

In [ ]:
def sem_population(values):
    """Population-SD SEM convention used in the released notebook."""
    # TODO
    raise NotImplementedError

def path_efficiency(path):
    """Straight-line endpoint distance divided by total sample-to-sample distance."""
    # TODO: require at least two samples; define behavior for a zero-length path.
    raise NotImplementedError

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
np.testing.assert_allclose(sem_population(x), np.std(x, ddof=0) / np.sqrt(4))

straight = np.array([[0.0, 0.0], [1.0, 0.0], [2.0, 0.0]])
detour = np.array([[0.0, 0.0], [0.0, 1.0], [2.0, 0.0]])
np.testing.assert_allclose(path_efficiency(straight), 1.0)
assert 0 < path_efficiency(detour) < 1
print("Metric primitive self-checks passed")

In [ ]:
def analyze_block(path, finger_columns):
    """Return trial-level acquisition, hold, rate, success and path metrics."""
    # TODO. Write pseudocode in the Markdown cell before implementation.
    # Required safeguards:
    # - align clocks;
    # - identify trials from targets;
    # - preserve failed trials for percent completed;
    # - exclude the final 500-ms hold from acquisition time;
    # - compute path efficiency only on successful trajectories.
    raise NotImplementedError

def aggregate_blocks(block_results):
    """Aggregate trial metrics and per-block target rates."""
    # TODO
    raise NotImplementedError

**Write pseudocode before implementing `analyze_block`:** [write here]

**Implementation questions**

1. Why is the final 500-ms hold excluded from acquisition time?
2. Why are failed trials retained for completion percentage but excluded from timing/path summaries?
3. Why is target rate averaged per block rather than computed as total successes divided by total time?
4. What changes when SEM uses `ddof=1` rather than `ddof=0`?
5. Which metric is most sensitive to one-sample trial-boundary errors?
6. What does path efficiency equal for a perfectly straight path? Can it exceed 1 because of floating-point noise?

**Answers:** [write here]

In [ ]:
# Block selections for the headline comparison. These are analytical choices,
# not discovered from the filenames automatically.
HEADLINE_2D_BLOCKS = [
    "20230321/RedisMat/t5_t5.2023.03.21_Data_RedisMat_20230321_162754_(18).mat",
    "20230321/RedisMat/t5_t5.2023.03.21_Data_RedisMat_20230321_163420_(19).mat",
    "20230323/RedisMat/t5_t5.2023.03.23_Data_RedisMat_20230323_135514_(7).mat",
    "20230323/RedisMat/t5_t5.2023.03.23_Data_RedisMat_20230323_140002_(8).mat",
    "20230323/RedisMat/t5_t5.2023.03.23_Data_RedisMat_20230323_140335_(9).mat",
    "20230406/RedisMat/t5_t5.2023.04.06_Data_RedisMat_20230406_144356_(15).mat",
    "20230406/RedisMat/t5_t5.2023.04.06_Data_RedisMat_20230406_144851_(16).mat",
]
HEADLINE_4D_BLOCKS = [
    "20230309/RedisMat/t5_t5.2023.03.09_Data_RedisMat_20230309_150822_(18).mat",
    "20230314/RedisMat/t5_t5.2023.03.14_Data_RedisMat_20230314_155042_(17).mat",
    "20230316/RedisMat/t5_t5.2023.03.16_Data_RedisMat_20230316_144512_(14).mat",
    "20230316/RedisMat/t5_t5.2023.03.16_Data_RedisMat_20230316_145004_(15).mat",
    "20230323/RedisMat/t5_t5.2023.03.23_Data_RedisMat_20230323_152402_(24).mat",
    "20230323/RedisMat/t5_t5.2023.03.23_Data_RedisMat_20230323_152739_(25).mat",
    "20230406/RedisMat/t5_t5.2023.04.06_Data_RedisMat_20230406_155957_(30).mat",
    "20230413/RedisMat/t5_t5.2023.04.13_Data_RedisMat_20230413_131021_(7).mat",
    "20230413/RedisMat/t5_t5.2023.04.13_Data_RedisMat_20230413_131337_(8).mat",
]

# TODO: analyze both groups, make the six-panel Fig. 1e plot, and save it.
# Your rounded anchors from the paper are:
# acquisition time 1.33 vs 1.98 s; rate 88 vs 64 targets/min;
# completion 98.1% vs 98.7%; path efficiency 0.718 vs 0.524.

**Statistical questions**

1. Reproduce the reported approximately 50% acquisition-time increase. State numerator and denominator.
2. Why might a trial-level independent-samples t-test understate uncertainty?
3. Recompute the comparison using block means. Does the interpretation change?
4. Identify every analysis choice required to obtain the published values.

**Answers:** [write here]

### Module 4 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 4:** Change the hold interval from 500 to 450 ms in a copy of the analysis. The student must predict which panels move, identify why, and restore the correct result.

# Module 5 - Neural dimensionality and Fig. 2a

**Learning objective:** Explain and implement the participation-ratio dimensionality measure.

For eigenvalues $\lambda_i$ of a covariance representation,

$$D = \frac{(\sum_i \lambda_i)^2}{\sum_i \lambda_i^2}.$$

In [ ]:
def participation_ratio(eigenvalues):
    """Return effective dimensionality from nonnegative eigenvalues."""
    # TODO: validate finiteness/nonnegativity and handle an all-zero input.
    raise NotImplementedError

In [ ]:
np.testing.assert_allclose(participation_ratio([1, 1, 1, 1]), 4.0)
np.testing.assert_allclose(participation_ratio([4, 0, 0, 0]), 1.0)
np.testing.assert_allclose(
    participation_ratio(np.array([2.0, 1.0]) * 10),
    participation_ratio([2.0, 1.0]),
)
print("Participation-ratio self-checks passed")

**Before applying to neural data**

1. Why is participation ratio invariant to multiplying all eigenvalues by a constant?
2. What does a value of 2.4 mean, and why need it not be an integer?
3. How do channel z-scoring, time selection, and trial selection affect the eigenvalue spectrum?
4. Why does more decoder output DOF not mathematically guarantee higher neural dimensionality?
5. Distinguish PCA component count chosen by a variance threshold from participation ratio.

**Answers:** [write here]

In [ ]:
def session_dimensionality(mat_paths, finger_columns, task_selection):
    """Reproduce one session's neural dimensionality from selected closed-loop bins."""
    # TODO: trace the authors' dimensionality function before writing code.
    # Document: selected bins, neural normalization, concatenation axis,
    # covariance/PCA convention, and eigenvalues passed to participation_ratio.
    raise NotImplementedError

# TODO: compute the three groups and recreate Fig. 2a.
# Rounded paper anchors: 2.4, 3.1, and 7.5.

**Interpretation questions**

1. The 4-DOF/two-target mean is more than twice the 2-DOF mean. Reconstruct the percentage above `2 x D_2D`.
2. Which data points in Fig. 2a are sessions and which are means?
3. Why is the one-target 4-DOF condition represented by only one point?
4. Give one neural and one behavioral explanation for higher dimensionality.
5. What additional data would distinguish these explanations?

**Answers:** [write here]

### Module 5 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 5:** Give spectra `[5, 5, 0, 0]` and `[9, 1, 0, 0]`. The student must order their dimensionalities without a calculator and explain the result geometrically.

# Module 6 - Cross-decoder prediction and Fig. 2c-d

**Learning objective:** Trace data through a released neural-network checkpoint and quantify agreement between online and offline velocity traces.

First inspect `NNDecoders.py` and one checkpoint. Draw the tensor shapes from three 50-ms input bins to the decoder output.

**Architecture questions**

1. What is shared across time before the fully connected layers?
2. Why are exactly three recent bins used?
3. Where are batch normalization, dropout, and ReLU applied?
4. What changes between two-output and four-output checkpoints?
5. Why should inference call evaluation mode?

**Answers / shape diagram:** [write here]

In [ ]:
def lagged_normalized_correlation(
    reference, prediction, max_lag=None, center=False
):
    """Return normalized cross-correlation and the lag of its maximum.

    center=False reproduces the released source statistic. Use
    center=True only as a separately labelled sensitivity analysis.
    """
    # TODO: validate inputs, optionally mean-center, define the energy
    # normalization and lag sign, reject zero-energy input, and
    # optionally restrict max_lag.
    raise NotImplementedError

In [ ]:
base = np.array([0.0, 1.0, 0.0, -1.0, 0.0])
cc, lag = lagged_normalized_correlation(base, base, max_lag=2, center=False)
np.testing.assert_allclose(cc, 1.0)
assert lag == 0

shifted = np.roll(base, 1)
cc_shift, lag_shift = lagged_normalized_correlation(
    base, shifted, max_lag=2, center=False
)
assert cc_shift > 0.99
assert abs(lag_shift) == 1

try:
    lagged_normalized_correlation(np.zeros(5), base, center=False)
except ValueError:
    pass
else:
    raise AssertionError("zero-energy input must raise ValueError")
print("Cross-correlation self-checks passed")

In [ ]:
def run_released_decoder(checkpoint_path, neural_bins, hyperparameters):
    """Load one checkpoint and return denormalized velocity predictions."""
    # TODO: instantiate the correct released class, apply the 256-wide
    # mask with exactly 192 active positions (do not slice the model
    # input to width 192), apply normalization, build 3-bin windows,
    # call eval/no_grad, and undo output gain/mean.
    raise NotImplementedError

# TODO: reproduce the Fig. 2c 30-second trace and five-block means in Fig. 2d.
# Rounded anchors: 0.69 and 0.68. These are the released, uncentered
# normalized cross-correlations (center=False). Report a centered
# sensitivity analysis separately if you compute one.

**Implementation and interpretation questions**

1. Why does the decoder retain a 256-wide input while its mask has exactly 192 active positions?
2. What happens if input normalization is applied after channel selection using the wrong axis?
3. Why is `map_location` important when loading a checkpoint on a computer without a compatible GPU?
4. Does a correlation of 0.68 imply unbiased predictions or correct amplitude? Explain.
5. Why average five block-level correlations rather than concatenate every block first?
6. How would a one-bin timing shift affect the zero-lag and maximum-lag results?

**Answers:** [write here]

### Module 6 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 6:** Without rerunning training, change the inference code so dropout remains active. The student must diagnose run-to-run variation and identify the missing model-state call.

# Module 7 - Directional SNR and channel count (Fig. 3b-c)

**Learning objective:** Decompose predicted velocity into intended-direction signal and orthogonal noise, then fit a power law across channel counts.

In [ ]:
def signal_and_noise_components(predicted, intended_unit):
    """Project sample-by-feature predictions parallel and orthogonal to intent."""
    # TODO: normalize/validate intended_unit, compute scalar projection,
    # reconstruct the parallel vector, and return parallel plus residual.
    raise NotImplementedError

def fit_power_law(channel_counts, dsnr_values, fit_fraction=0.75):
    """Fit log(dSNR) = intercept + exponent * log(channels)."""
    # TODO: use the highest fit_fraction of counts, return exponent and R^2.
    raise NotImplementedError

In [ ]:
predicted = np.array([[2.0, 1.0], [1.0, -2.0]])
intended = np.array([1.0, 0.0])
parallel, residual = signal_and_noise_components(predicted, intended)
np.testing.assert_allclose(parallel, [[2.0, 0.0], [1.0, 0.0]])
np.testing.assert_allclose(residual, [[0.0, 1.0], [0.0, -2.0]])

counts = np.array([5, 10, 20, 40, 80], dtype=float)
synthetic_dsnr = 0.2 * counts ** 0.4
exponent, r2 = fit_power_law(counts, synthetic_dsnr, fit_fraction=1.0)
np.testing.assert_allclose(exponent, 0.4, atol=1e-12)
np.testing.assert_allclose(r2, 1.0, atol=1e-12)
print("dSNR primitive self-checks passed")

**Before the expensive sweep**

1. Define directional SNR in words and with a formula.
2. Why is the intended vector inferred from targets rather than observed finger movement?
3. Why must channel subsets be sampled repeatedly?
4. The Methods say 25 subsets, while the notebook uses 50. Which will you preregister and report?
5. Why is the fit restricted to the highest 75% of channel counts?
6. What does an exponent below 0.5 suggest under the paper's noise assumptions?

**Answers:** [write here]

In [ ]:
# Stage A: reduced development run. This should finish quickly and must pass before the full run.
DEVELOPMENT_COUNTS = np.array([5, 25, 50, 100, 192])
DEVELOPMENT_SUBSETS = 5

def directional_snr_channel_sweep(session_data, channel_counts, subsets, rng):
    """Fit velocity mappings on random channel subsets and return dSNR curves."""
    # TODO: make every random choice come from rng; do not use global state.
    raise NotImplementedError

# Exact source mode uses NumPy's legacy MT19937 stream, not default_rng:
# source_rng = np.random.RandomState(0)
# Create it once immediately before the first session, then pass the same
# continuing object through every session in released-notebook call order.
# A Generator-based implementation is a valid, separately labelled
# sensitivity analysis, but it will not reproduce the exact source draws.

# Stage B: optional/final full reproduction after Stage A passes.
PAPER_COUNTS = np.array([
    5, 10, 19, 29, 38, 48, 58, 67, 77, 86, 96,
    106, 115, 125, 134, 144, 154, 163, 173, 182, 192,
])
PAPER_SUBSETS_IN_NOTEBOOK = 50

**Expected full-run checks (do not tune your code to these):** the three plotted exponents round to 0.34, 0.38, and 0.43; all fitted $R^2$ values round to 0.99 or 1.00.

**Interpretation questions**

1. Why does increasing dSNR through 192 channels not prove improvement beyond 192?
2. Identify a possible leakage issue when adjacent 50-ms samples are randomly partitioned across folds.
3. Design a trial-grouped cross-validation alternative. Predict how it may change dSNR.
4. The generated blue fit annotation is clipped. Why is this a presentation defect rather than a numerical mismatch?

**Answers:** [write here]

### Module 7 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 7:** Supply identical synthetic channels. The student must predict whether adding them should improve effective information and diagnose why a naive regression may nevertheless appear to improve.

# Module 8 - Open-loop nine-class classification (Extended Data Fig. 1b-c)

**Learning objective:** Port the MATLAB diagonal-linear classifier and reproduce two confusion matrices.

Use the two files in `Data/OpenLoopData`. Begin by confirming 200 trials, 192 selected channels, and 40 common 50-ms bins.

In [ ]:
def diaglinear_predict(train_x, train_y, test_x, variance_floor=1e-12):
    """Pooled diagonal-covariance Gaussian classifier with class priors."""
    # TODO: calculate class means, pooled within-class diagonal variance,
    # priors, log discriminants, and predicted labels.
    raise NotImplementedError

def row_normalized_confusion(actual, predicted, labels):
    """Return counts and a row-normalized confusion matrix."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(3)
class0 = rng.normal(loc=-2.0, scale=0.2, size=(30, 3))
class1 = rng.normal(loc=+2.0, scale=0.2, size=(30, 3))
train_x = np.vstack([class0[:20], class1[:20]])
train_y = np.array([0] * 20 + [1] * 20)
test_x = np.vstack([class0[20:], class1[20:]])
test_y = np.array([0] * 10 + [1] * 10)
pred = diaglinear_predict(train_x, train_y, test_x)
assert np.mean(pred == test_y) > 0.95
print("Diagonal-linear synthetic check passed")

**Implementation questions**

1. What independence assumption makes the covariance diagonal?
2. Why use a pooled variance rather than one variance per class?
3. Why must the MATLAB-compatible stratified partition be seeded carefully?
4. What is the difference between overall accuracy and the mean diagonal of a row-normalized confusion matrix?
5. Why can a 150-ms window retain useful accuracy despite containing only three bins?
6. Which pairs of finger movements are most frequently confused, and what neurophysiological explanation is plausible?

**Answers:** [write here]

In [ ]:
def run_open_loop_reproduction(data_root, window_bins, rng):
    """Parse trials, create MATLAB-compatible folds, classify, and summarize.

    Pass one continuing RNG so sequential window analyses consume the
    same stream just as the released MATLAB script does.
    """
    # TODO: reproduce the released MATLAB analysis in Python.
    raise NotImplementedError

# Exact source mode:
# matlab_rng = np.random.RandomState(5489)  # MATLAB rng('default')
# result_2s = run_open_loop_reproduction(DATA_ROOT, 40, matlab_rng)
# result_150ms = run_open_loop_reproduction(DATA_ROOT, 3, matlab_rng)
# Do not reset the stream between the two calls.
# TODO: run window_bins=40 (2 s) and window_bins=3 (150 ms), then plot.
# Published/reproduced accuracy anchors: 84% and 79%.

### Module 8 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 8:** Shuffle training labels but not test labels. The student must predict the confusion matrix, run it, and explain any residual above-chance result.

# Module 9 - Quadcopter exemplar path and Fig. 4c

**Learning objective:** Parse the released 3-D path, reproduce the full trajectory and four lap slices, and state the limit of the available flight data.

In [ ]:
FLIGHT_FILE = DATA_ROOT / (
    "20230712/RedisMat/"
    "t5_t5.2023.07.12_Data_RedisMat_20230712_151106_(14).mat"
)

def make_ring(center, radius, points=360):
    """Return coordinates for one vertical circular ring."""
    # TODO: document the ring plane and coordinate order.
    raise NotImplementedError

def plot_flight_path(mat_path):
    """Recreate the full path and four overlapping MATLAB lap slices."""
    # TODO: load x/y/z, plot -z for elevation, reproduce the view,
    # add rings, and use the four published/source lap boundaries.
    raise NotImplementedError

In [ ]:
# After implementing, these checks should pass.
flight_raw = loadmat(FLIGHT_FILE)
# The released file stores coordinates as three separate column vectors.
# TODO: uncomment after inspecting and confirming these manifest keys.
# positions = np.column_stack([
#     np.asarray(flight_raw["x"]).ravel(),
#     np.asarray(flight_raw["y"]).ravel(),
#     np.asarray(flight_raw["z"]).ravel(),
# ])
# assert positions.shape == (81601, 3)
# assert abs((positions.shape[0] - 1) / 500 - 163.2) < 1e-12

**Questions**

1. Why is `-z` plotted as elevation?
2. Why do adjacent lap slices share one boundary sample?
3. Are the ring coordinates measured from data or hard-coded by the plotting script?
4. Which visual differences can arise from MATLAB and Matplotlib 3-D camera projections?
5. The paper reports 12 obstacle-course flights, but only one 163-s path is released. Which statements can and cannot be checked?
6. Why would reconstructing aggregate mean completion time from this exemplar be invalid?

**Answers:** [write here]

### Module 9 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Instructor checkpoint 9:** Remove the sign change on z and alter one lap boundary. The student must identify both errors from the plot and sample counts.

# Module 10 - Validation, discrepancies, and final report

**Learning objective:** Distinguish numerical agreement, visual agreement, and scientific evidence.

Complete the table with full-precision computed values, paper-rounded values, absolute/relative differences, and a pass/fail tolerance chosen before looking at the answer.

| Result | Your value | Paper value | Tolerance | Pass? | Explanation |
|---|---:|---:|---:|---|---|
| 2D successful trials | | 529 | | | |
| 4D successful trials | | 524 | | | |
| 2D acquisition time (s) | | 1.33 | | | |
| 4D acquisition time (s) | | 1.98 | | | |
| Dimensionality: 2D | | 2.4 | | | |
| Dimensionality: 4D one-target | | 3.1 | | | |
| Dimensionality: 4D two-target | | 7.5 | | | |
| Cross-decoder means | | 0.69 / 0.68 | | | |
| dSNR exponents | | 0.34 / 0.38 / 0.43 | | | |
| Open-loop accuracy | | 84% / 79% | | | |
| Flight samples / duration | | 81,601 / 163 s | | | |

### Required discrepancy log

For at least four discrepancies, record:

| Observation | Numerical, code, data, or presentation? | Cause | Scientific impact | Proposed fix |
|---|---|---|---|---|
| Fig. 1c filename overwrite | | | | |
| Methods says 25 subsets; notebook uses 50 | | | | |
| Fig. 3c clipped annotation | | | | |
| `LoadData` trial interpolation bug | | | | |
| One additional discrepancy | | | | |

### Final synthesis questions

1. Which result is most robust to reasonable implementation choices? Which is least robust?
2. Identify one result that is numerically reproduced but does not independently validate the paper's biological claim.
3. Propose a hierarchical or repeated-measures alternative to one trial-level test.
4. Propose one new analysis that can be answered with the released data and one that cannot.
5. If a different package version changes the last decimal but not the paper-rounded value, how should you report it?
6. Write a 150-word replication verdict that includes scope, successes, limitations, and provenance.

**Answers:** [write here]

## Final submission checklist

- [ ] I created an isolated Conda or `venv` environment and recorded the exact commands.
- [ ] `python -m pip check` passes, and the notebook reports the intended environment's `sys.executable`.
- [ ] Notebook runs from a clean kernel in the documented environment on my operating system.
- [ ] Every required function has a purpose statement, input/output contract, test, and failure behavior.
- [ ] Figures 1c, 1e, 2a, 2c-d, 3b-c, ED 1b-c, and 4c are saved.
- [ ] Full-precision results are exported to JSON or CSV.
- [ ] Preregistration, provenance, discrepancy log, and every module's AI/search record are complete.
- [ ] At least one commit exists per module.
- [ ] Final report distinguishes computational reproduction from experimental replication.
- [ ] Student can defend any randomly selected function and modify it on a new test case.

### Module 10 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Final technical oral defense (20 minutes)**

1. Instructor selects one data-loading/alignment function and one analysis function at random.
2. Student states purpose, shapes, assumptions, complexity, and one failure mode.
3. Student predicts the result of a small code modification before running it.
4. Student interprets one reproduced figure without reading notebook text.
5. Student names one claim the public artifacts cannot establish.

# Module 11 - Willsey Lab application-readiness capstone

**Learning objective:** Translate verified technical work into a confident, ethical, and evidence-backed research application.

Begin only after completing the technical defense. Read `student_capstone.md`, then recheck the official lab, faculty, and joining pages during the week you prepare the application. The 2025 paper reflects work conducted during Dr. Willsey's Stanford period; the current Willsey Lab is at the University of Michigan. Treat the paper and the current lab as connected but not identical contexts.

As of 7 August 2026, the official joining page describes graduate, postdoctoral, and research-specialist routes but does not name a specific undergraduate category. Do not imply that an undergraduate vacancy exists unless a current source or direct reply confirms it.

**Application-readiness questions**

1. In two sentences, what does the Willsey Lab currently study? Cite official sources and the date checked.
2. Which three completed project artifacts best demonstrate fit, and what does each fail to demonstrate?
3. What exactly did you reproduce, and what did you not reproduce?
4. Which technical decision in the analysis required the most judgment?
5. Describe one discrepancy without implying misconduct. What was its numerical and scientific impact?
6. Which follow-up analysis would you start first, and why is its validation plan stronger?
7. Why this lab rather than a generic neurotechnology lab?
8. What could you contribute during an initial research period, and what would you need to learn?
9. How would you handle public human neural data responsibly?
10. What should you do if no undergraduate role is listed?
11. How did AI or search assistance enter the project, and how did you verify it?
12. Write and rehearse a two-minute project pitch.

**Answers:** [write here]

### Application portfolio and evidence-to-claim audit

Prepare these from work you actually completed. Keep private application drafts separate from the public technical portfolio. Do not publish instructor materials, raw AI transcripts, credentials, or source data without confirming authorization and licensing.

- [ ] Public-safe project README with provenance, environment setup, figure gallery, numerical results, limitations, and a next question.
- [ ] One-page technical brief.
- [ ] Five-slide, five-minute research talk.
- [ ] One project title and two or three evidence-based CV bullets.
- [ ] Tailored inquiry or cover-letter draft naming the exact route sought without assuming an unlisted opening.
- [ ] Two-minute spoken project pitch.
- [ ] One-page follow-up proposal with hypothesis, unit of analysis, leakage controls, statistic, uncertainty, alternatives, and first-week plan.

| Proposed application claim | Supporting artifact | Required qualifier | Stronger claim not supported |
|---|---|---|---|
| [write] | [cell/figure/test/commit/report] | [write] | [write] |
| [write] | [cell/figure/test/commit/report] | [write] | [write] |
| [write] | [cell/figure/test/commit/report] | [write] | [write] |

**Target application route:** [write here]  
**Official joining page checked on:** [YYYY-MM-DD]  
**Portfolio location:** [write here]  
**One current lab theme connected to my proposed next analysis:** [write here]

### Insightful questions to ask Dr. Willsey

Draft at least five candidate questions after reading the current lab pages, two recent projects or papers, and your own discrepancy/proposal notes. A good question seeks scientific judgment that is not already stated online; it is grounded in specific evidence, concise enough to ask aloud, and leaves room for a follow-up. It should not be a disguised speech about your accomplishments.

| Candidate question | Current source or project that motivated it | Why the answer would matter | Why it is not already answered publicly | Possible follow-up |
|---|---|---|---|---|
| [write] | [URL/citation and date] | [write] | [write] | [write] |
| [write] | [URL/citation and date] | [write] | [write] | [write] |
| [write] | [URL/citation and date] | [write] | [write] | [write] |
| [write] | [URL/citation and date] | [write] | [write] | [write] |
| [write] | [URL/citation and date] | [write] | [write] | [write] |

Eliminate questions answered by the website, yes/no questions, generic questions that could be asked of any lab, adversarial wording, and questions that assume an opening. Select the best **two**: one scientific question and one question about research practice, mentorship, or a useful first contribution.

**Selected scientific question and why:** [write here]  
**Selected research-practice/mentorship question and why:** [write here]

### Module 11 - AI and search record

Complete this short record before the checkpoint. It is an attachment record, not a detailed diary.

**If you used an AI assistant:** export or copy the conversation and paste it below with prompts and answers in their original order. Redact only passwords, tokens, private information, or unrelated personal material, and mark any removal as `[REDACTED]`.

**Pasted AI conversation:**  
[Paste exported conversation here, or write "AI not used."]

**If you used a search engine:** list the exact search terms in the order used, one per line.

**Search terms:**
- [Paste search term, or write "Search engine not used."]

**Notebook step or function affected:** [one short line, or "None"]  
**Independent check:** [one sentence or test-cell reference]

**Application-readiness mock interview (30 minutes)**

1. Student gives the two-minute project pitch without notes.
2. Student explains why the proposed next analysis fits one currently verified lab theme.
3. Student answers one technical, one research-fit, and one human-data ethics question.
4. Student identifies one skill that remains to be learned and gives a concrete learning plan.
5. Student revises one overclaimed CV or email sentence into an evidence-bounded statement.
6. Student asks one of the two selected questions, explains what evidence motivated it, and gives a natural follow-up based on a plausible answer.